# Exploration of various ways to identify the reaches from the distance-to-kinect data

In [ ]:
### imports and globals 

import pyxdf

# for the tests
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import butter, sosfiltfilt, find_peaks

%matplotlib qt

# optional visualizations within functions
do_visualize = True

## Functions to manage rearm xdf files 

In [ ]:
def print_streams_types_and_names(fullFname_or_streams):
    """Print the names and types of all streams in the xdf file or in the streams list"""

    if isinstance(fullFname_or_streams, str):
        xdf_data, header = pyxdf.load_xdf(filename=fullFname_or_streams, verbose=False)
    elif (
        isinstance(fullFname_or_streams, list)
        and all(isinstance(x, dict) for x in fullFname_or_streams)
        and all("info" in x for x in fullFname_or_streams)
    ):
        xdf_data = fullFname_or_streams
    else:
        raise ValueError("The first argument must be a filename or a list of streams")

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, searched_stream_type, searched_stream_names):
    """Get the stream of type 'searched_stream_type' with name in 'searched_stream_names' in the xdf_data"""

    if not isinstance(
        searched_stream_names, list
    ):  # if we get a string (only one name)
        searched_stream_names = [searched_stream_names]

    found_streams = []
    for stream in xdf_data:
        stream_type = stream["info"]["type"][0]
        if searched_stream_type == stream_type:
            stream_name = stream["info"]["name"][0]
            for searched_stream_name in searched_stream_names:
                if searched_stream_name == stream_name:
                    found_streams.append(stream)

    if not found_streams:
        # msg = f" Stream not found. Searched in [{searched_stream_type}:{searched_stream_names}]."
        # print(msg)
        return None

    if len(found_streams) > 1:
        found_streams_names = [stream["info"]["name"][0] for stream in found_streams]
        msg = f"Found multiple streams: [{searched_stream_type},{found_streams_names}]."
        raise ValueError(msg)

    return found_streams[0]


def get_kinect_channel_indexes(kinect_mocap, searched_label):
    """Get the indexes of all kinect channels containing searched_label in their name"""
    channel_indexes = []
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if searched_label in current_name:
            channel_indexes.append(i)
    if channel_indexes == []:
        raise ValueError(f"Pattern {searched_label} not found in the kinect mocap data")

    return channel_indexes


def get_kinect_channel_index(kinect_mocap, channel_name):
    """Get the index of one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    return channel_index


def get_kinect_channel_data(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = get_kinect_channel_index(kinect_mocap, channel_name)
    channel_data = kinect_mocap["time_series"][:, channel_index]
    return channel_data


def interpolate_to_constant_time_step(t, x, dt=0.033):
    """Interpolate the data to a constant time step"""

    n_columns = x.shape[1] if x.ndim > 1 else 1

    t_new = np.arange(t[0], t[-1], dt)

    if x.ndim < 2:
        x_new = np.interp(t_new, t, x)
    else:
        x_new = np.zeros((len(t_new), n_columns))
        for i in range(n_columns):
            x_new[:, i] = np.interp(t_new, t, x[:, i])

    return x_new, t_new

## Load the xdf file

In [ ]:
xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3/ReArm_C1P02_20210715_V3_Reaching/ReArm_C1P07_20211116_V3_r.xdf"
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # no mouse data --> eventIDE
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Reaching/task-V2_Reach.xdf"  # no mouse data --> eventIDE
xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"  # old data
xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros TODO
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P45/V2/Reaching/task-V2_Reach.xdf"  # no mouse data --> eventIDE
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V2/Reaching/C1P31_RauMic_20230331_2_r.xdf"  # wrong correction? look good
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching/task-V3_Reach.xdf"  # no mouse data --> eventIDE
# xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf"  # Kinect markers with 1 value that is empty

xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P05/ReArm_C1P05_20210514_V1/ReArm_C1P05_20210514_V1_Reaching/ReArm_C1P05_20210621_V1_r.xdf"  # Kinect markers with 1 value that is empty
xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231009_V1/ReArm_C1P38_20231009_V1_Reaching/ReArm_C1P38_20231009_V1_r.xdf"  # Kinect markers with 1 value that is empty

xdf_data, header = pyxdf.load_xdf(
    filename=xdf_fullFname,
    select_streams=[
        {"type": "MoCap"},
        {"type": "Markers"},
    ],
    synchronize_clocks=True,
    dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    verbose=False,
)

# TODO : we should locate EARLY (just after loading?) the interpolation of the mocap data
# because kinect and mouse are NOT a continuous stream (holes in the data)

print_streams_types_and_names(xdf_data)

## Make the time correction (if needed)

In [ ]:
def get_time_correction(xdf_fullFname):
    """Get the time correction from the xdf file name"""
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    try:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)
    except FileNotFoundError:
        time_correction = np.float64(0)
    return time_correction


time_correction = get_time_correction(xdf_fullFname)
print(f"Time correction: {time_correction} s")

# make the time correction
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

if kinect_mocap:
    kinect_mocap["time_stamps"] = kinect_mocap["time_stamps"] + time_correction
if kinect_markers:
    kinect_markers["time_stamps"] = kinect_markers["time_stamps"] + time_correction

### Remove the kinect samples filled with zeros
This has to be done before any processing of the kinect data, as this is to fix a bug due to the kinect. 

In [ ]:
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
mouse_mocap = get_stream(xdf_data, "MoCap", "Mouse")

if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]
    # find the indexes of kinect data that are filled with zeros
    zero_rows = np.all(kinect_data == 0, axis=1)
    zero_rows_indices = np.where(zero_rows)[0]
    print(f"Found {len(zero_rows_indices)} rows filled with only zeros")
    if len(zero_rows_indices) > 0:
        # remove the zero rows from the data
        kinect_data = np.delete(kinect_data, zero_rows_indices, axis=0)
        kinect_t = np.delete(kinect_t, zero_rows_indices, axis=0)

        WristRight_Z_before = get_kinect_channel_data(kinect_mocap, "WristRight_Z")
        kinect_t_before = kinect_mocap["time_stamps"]

        # modify the original data
        kinect_mocap["time_series"] = kinect_data
        kinect_mocap["time_stamps"] = kinect_t

        WristRight_Z = get_kinect_channel_data(kinect_mocap, "WristRight_Z")

        # plot WristRight_Z
        plt.figure()
        plt.plot(kinect_t_before, WristRight_Z_before, ".", label="before")
        plt.plot(kinect_t, WristRight_Z, ".", label="after")
        plt.title("WristRight_Z: before and after removing zero rows")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.show()

        del WristRight_Z_before
        del kinect_t_before

    # clean globals
    del zero_rows
    del zero_rows_indices

## Interpolate the Mocap data
This is mandatory because the kinect and mouse data are produced by the computer: the sampling rate is not waranted to be constant (samples are forgetten... sometimes). 

In [ ]:
def get_joint_norm(kinect_mocap, joint_name):
    """Get the norm of the joint position"""
    joint_X = get_kinect_channel_data(kinect_mocap, joint_name + "_X")
    joint_Y = get_kinect_channel_data(kinect_mocap, joint_name + "_Y")
    joint_Z = get_kinect_channel_data(kinect_mocap, joint_name + "_Z")
    joint_Norm = np.sqrt(joint_X**2 + joint_Y**2 + joint_Z**2)
    return joint_Norm


## proceed step by step with visualization
if kinect_mocap:

    # re-read the data (that may have been modified)
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]

    # # interpolate all the kinect data to a constant time step

    WristLeft_Norm = get_joint_norm(kinect_mocap, "WristLeft")
    WristRight_Norm = get_joint_norm(kinect_mocap, "WristRight")

    reach_left, reach_t = interpolate_to_constant_time_step(kinect_t, WristLeft_Norm)
    reach_right, reach_t = interpolate_to_constant_time_step(kinect_t, WristRight_Norm)

    # Test : interpolate 2 columns at a time
    w_lr = np.vstack((WristLeft_Norm, WristRight_Norm)).T
    w_lr_, reach_t_ = interpolate_to_constant_time_step(kinect_t, w_lr)
    reach_left_ = w_lr_[:, 0]
    reach_right_ = w_lr_[:, 1]
    # assert that the result is the same
    assert reach_left.shape == reach_left_.shape
    assert reach_right.shape == reach_right_.shape
    assert reach_t.shape == reach_t_.shape
    assert len(reach_left) == len(reach_right)
    assert len(reach_left) == len(reach_t)
    assert len(reach_right) == len(reach_t)
    assert np.allclose(reach_t, reach_t_)
    assert np.allclose(reach_left, reach_left_)
    assert np.allclose(reach_right, reach_right_)
    print("Interpolation multiple column test passed")

    # plot the raw and interpolated data
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(kinect_t, WristLeft_Norm, ".", label="Left wrist", color="b")
    ax.plot(kinect_t, WristRight_Norm, ".", label="Right wrist", color="k")
    ax.plot(reach_t, reach_left, label="Left wrist interpolated", color="b", alpha=0.2)
    ax.plot(
        reach_t, reach_right, label="Right wrist interpolated", color="k", alpha=0.2
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()
    plt.show()

    # delete the variables that MUST be recomputed
    del kinect_t
    del kinect_data
    del WristLeft_Norm
    del WristRight_Norm
    del reach_left
    del reach_right
    del reach_t
    del w_lr
    del w_lr_
    del reach_t_


#######################
## real job is below...
def resample_stream(stream):
    """Resample the stream to a constant time step"""

    # get the type and name of the stream
    stream_type = stream["info"]["type"][0]
    stream_name = stream["info"]["name"][0]
    if stream_type != "MoCap":
        return stream

    # check if the stream was resampled before
    if "resampled" in stream["info"]:
        print(f"Stream '{stream_name}' was already resampled")
        return stream

    t = stream["time_stamps"]
    data = stream["time_series"]
    # resample the data to a constant time step
    t_step = 1 / 30  # 30 Hz typical for kinect
    data, t = interpolate_to_constant_time_step(t, data, dt=t_step)
    # modify the original data
    stream["time_series"] = data
    stream["time_stamps"] = t
    stream["info"]["desc"][0]["channels"][0]["channel"][0]["nominal_srate"] = 30
    stream["info"]["desc"][0]["channels"][0]["channel"][0]["preferred_srate"] = 30
    # add a new field to the info
    stream["info"]["resampled"] = ["True"]

    return stream


if mouse_mocap:
    mouse_mocap = resample_stream(mouse_mocap)


if kinect_mocap:
    kinect_mocap = resample_stream(kinect_mocap)

    # re-read the data (that may have been modified)
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]

## Get the markers 

The markers are necessary to identify the *zones of interest* within which the reaches are expected to occur. 
Outside these zones, the data is not relevant for the analysis, and it adds a lot of noise to the data.

Each zone has a *start* and *end* marker, but the label of the markers differs if the sequence was generated by : 

- the software **LSL-Mouse**: stream `mouse_to_nic_markers`
    - start = `"[111]"`  
    - stop = `"[100]"` 

- the software **event-IDE**: stream `event_to_nic_markers`
    - start = `"[100]"`, but we have to keep only the first start in the sequence before each stop
    - stop = `"[75]"`


The streams are loaded from the xdf file using the function `get_stream`:
``` python
# get the markers streams from the xdf file
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])
event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"]) 
```
``` python

### Function to get the markers of the reach zones in the data

In [ ]:
def find_marker_indexes(marker_name, markers_data):
    """Find the indexes of marker_name in markers_data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return np.array(marker_index_list)


def get_coherent_start_stop_times(starts, stops):
    """Get the coherent start and stop times from the start and stop times"""

    # make an array of start, 0 and stop, 1
    start = np.zeros(
        len(starts),
        dtype=[("time", float), ("type", int)],
    )
    start["time"] = starts
    start["type"] = 0
    stop = np.zeros(
        len(stops),
        dtype=[("time", float), ("type", int)],
    )
    stop["time"] = stops
    stop["type"] = 1
    start_and_stop = np.concatenate((start, stop))
    start_and_stop = np.sort(start_and_stop, order="time")

    for i in range(len(start_and_stop) - 1):
        # keep only the last start in case of multiple contiguous start
        if start_and_stop[i]["type"] == 0 and start_and_stop[i + 1]["type"] == 0:
            start_and_stop[i + 1]["type"] = -1
        # keep only the first stop in case of multiple contiguous stop
        if start_and_stop[i]["type"] == 1 and start_and_stop[i + 1]["type"] == 1:
            start_and_stop[i + 1]["type"] = -1

    # ensure that the first is a start and the last is a stop
    if start_and_stop[0]["type"] == 1:
        start_and_stop[0]["type"] = -1
    if start_and_stop[-1]["type"] == 0:
        start_and_stop[-1]["type"] = -1

    # clean the start_and_stop array
    start_and_stop_ok = start_and_stop[start_and_stop["type"] != -1]

    starts_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 0]
    stops_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 1]

    if len(starts_corrected) != len(stops_corrected):
        raise ValueError(
            f"Number of starts ({len(starts_corrected)}) and stops ({len(stops_corrected)}) are not equal"
        )

    return starts_corrected, stops_corrected


def get_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop times from mouse_to_nic_markers"""

    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find_marker_indexes("[111]", mouse_to_nic_markers_data)
    stop_marker_index_list = find_marker_indexes("[100]", mouse_to_nic_markers_data)
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    start_times_from_mouse_to_nic_markers, stop_times_from_mouse_to_nic_markers = (
        get_coherent_start_stop_times(
            start_times_from_mouse_to_nic_markers,
            stop_times_from_mouse_to_nic_markers,
        )
    )

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_start_stop_times_from_event_ide_TONIC(event_ide_tonic):
    """get the start and stop times from event_ide_tonic"""

    event_ide_markers_data = event_ide_tonic["time_series"]
    event_ide_markers_time = event_ide_tonic["time_stamps"]

    if not isinstance(event_ide_markers_data[0], list):
        event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

    start_marker_index_list = find_marker_indexes("[100]", event_ide_markers_data)
    stop_marker_index_list = find_marker_indexes("[75]", event_ide_markers_data)

    start_times_from_event_ide_tonic = event_ide_markers_time[start_marker_index_list]
    stop_times_from_event_ide_tonic = event_ide_markers_time[stop_marker_index_list]

    # keep only the first start time for each stop time
    previous_stop_time = 0
    good_start_times = []
    for stop_time in stop_times_from_event_ide_tonic:
        possible_start_times = start_times_from_event_ide_tonic[
            start_times_from_event_ide_tonic < stop_time
        ]
        possible_start_times = possible_start_times[
            possible_start_times > previous_stop_time
        ]
        start_time = possible_start_times[0] if len(possible_start_times) > 0 else None
        good_start_times.append(start_time)
        previous_stop_time = stop_time
    good_start_times = np.array(good_start_times)
    good_start_times = good_start_times[
        good_start_times != None  # noqa: E711
    ]  # should be useless...

    good_start_times, stop_times_from_event_ide_tonic = get_coherent_start_stop_times(
        good_start_times,
        stop_times_from_event_ide_tonic,
    )

    return (
        good_start_times,
        stop_times_from_event_ide_tonic,
    )


def get_start_stop_times_from_mouse_markers(mouse_markers):
    """get the start and stop times from mouse_markers"""

    # NOTE: alternative way to get the start and stop times
    # here used to check the consistency with the mouse_to_nic_markers

    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]

    start_marker_index_list = find_marker_indexes(
        "DoCycleChange:DoRecord", mouse_markers_data
    )
    stop_marker_index_list = find_marker_indexes(
        "DoCycleChange:DoPause", mouse_markers_data
    )
    start_times_from_mouse_markers = mouse_markers_time[start_marker_index_list]
    stop_times_from_mouse_markers = mouse_markers_time[stop_marker_index_list]

    start_times_from_mouse_markers, stop_times_from_mouse_markers = (
        get_coherent_start_stop_times(
            start_times_from_mouse_markers,
            stop_times_from_mouse_markers,
        )
    )

    return (
        start_times_from_mouse_markers,
        stop_times_from_mouse_markers,
    )


def print_start_stop_times(start_stop_times):
    """Print the start and stop times from the (start_times, stop_times) tuple of lists"""

    start_times, stop_times = start_stop_times
    for i in range(len(start_times)):
        print(
            f"{i:02d}: {start_times[i]:8.2f} -> {stop_times[i]:8.2f}, Duration: {stop_times[i] - start_times[i]:5.2f}s"
        )

### Test the functions 

In [ ]:
## Test the functions

event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"])
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])

if mouse_to_nic_markers:
    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_mouse_to_nic_markers(
        mouse_to_nic_markers
    )
    print("Mouse to NIC markers:")
    print_start_stop_times((start_t, stop_t))
    # for the assert
    start_t_nic = start_t
    stop_t_nic = stop_t

if event_to_nic_markers:
    event_to_nic_markers_data = event_to_nic_markers["time_series"]
    event_to_nic_markers_time = event_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_event_ide_TONIC(event_to_nic_markers)
    print("Event IDE to NIC markers:")
    print_start_stop_times((start_t, stop_t))

# NOTE: To verify that we get the same start and stop times from the mouse markers and the mouse to nic markers
mouse_markers = get_stream(xdf_data, "Markers", ["Mouse", "Mouse-Markers"])
if mouse_markers:
    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_mouse_markers(mouse_markers)

    # assert that the start and stop times are the same
    start_t_mouse = start_t
    stop_t_mouse = stop_t
    assert len(start_t_mouse) == len(start_t_nic)
    assert len(stop_t_mouse) == len(stop_t_nic)
    assert np.allclose(start_t_mouse, start_t_nic)
    assert np.allclose(stop_t_mouse, stop_t_nic)
    print("Mouse markers and NIC markers are coherent :-)")


if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    WristRight_X = get_kinect_channel_data(kinect_mocap, "WristRight_X")
    WristRight_Y = get_kinect_channel_data(kinect_mocap, "WristRight_Y")
    WristRight_Z = get_kinect_channel_data(kinect_mocap, "WristRight_Z")

    WristLeft_X = get_kinect_channel_data(kinect_mocap, "WristLeft_X")
    WristLeft_Y = get_kinect_channel_data(kinect_mocap, "WristLeft_Y")
    WristLeft_Z = get_kinect_channel_data(kinect_mocap, "WristLeft_Z")

    WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
    WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)
    print(f"kinect_mocap['time_series'] shape: {kinect_mocap['time_series'].shape}")


if event_to_nic_markers:
    event_markers_data = event_to_nic_markers["time_series"]
    event_markers_time = event_to_nic_markers["time_stamps"]
    print(f"Event markers data shape: {event_markers_data.shape}")
    print(f"Event markers time shape: {event_markers_time.shape}")

### Low pass filter the data
We choose a cutoff frequency of 0.5 Hz, which corresponds to a time constant of 2 seconds = typical time of a reach.

In [ ]:
def butter_lowpass(cutoff, fs, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    sos = butter(
        order,
        normal_cutoff,
        btype="low",
        output="sos",  # recommended for numerical stability by scipy
    )
    return sos


def lowpass_filter(t, data):
    """Apply a lowpass filter to the data"""
    # check that the sampling period is constant
    dt = np.mean(np.diff(t))
    if not np.allclose(np.diff(t), dt):
        raise ValueError("The time vector do not have a constant sampling period")

    fs = 1 / dt  # sample rate, Hz
    cutoff = 0.5  # desired cutoff frequency of the filter, Hz
    order = 4  # order of the filter
    sos = butter_lowpass(cutoff, fs, order=order)
    filtered_data = sosfiltfilt(sos, data)
    return filtered_data

### Functions to identify the reaches
Includes debug options to visualize the data and the identified reaches.

In [ ]:
def index_of_last_negative_velocity_before_peak(t, velocity, t_end_i=None):
    """Get the index of the last negative velocity before the velocity peak"""

    if t_end_i is None:
        t_end_i = np.argmin(velocity)
    t_end = t[t_end_i]
    positive_before_t_end = velocity[t < t_end] > 0
    index_before_t_end = np.where(positive_before_t_end)[0]
    if len(index_before_t_end) == 0:
        return -1
    else:
        return max(index_before_t_end)


def get_one_reach(t, pos, t_end_i, do_plot=False):
    """
    get one reach from the position and time data
    """
    pos = np.array(pos)
    t = np.array(t)

    velocity = np.gradient(pos, t)
    f_velocity = lowpass_filter(t, velocity)

    # we know that the reach end
    t_end = t[t_end_i]
    # we look for the reach start
    t_beg_i = index_of_last_negative_velocity_before_peak(t, f_velocity, t_end_i)
    t_beg = t[t_beg_i]

    # verify that t_beg_i is not too close to the t_end_i
    if t_end_i - t_beg_i < 15:  # half a second
        t_beg_i = index_of_last_negative_velocity_before_peak(t, f_velocity, t_beg_i)

    # get the mask arround t_beg_*_i +/- 10
    t_beg_mask = range(t_beg_i - 10, t_beg_i + 10)
    t_end_mask = range(t_end_i - 10, t_end_i + 10)

    # we will return the median of the following positions
    beg_position = pos[t_beg_mask]
    end_position = pos[t_end_mask]

    # verify that the reach distance is larger than 0.05 m
    if np.abs(np.median(end_position) - np.median(beg_position)) < 0.05:
        return None

    if do_plot:

        def plot_one_sub(ax):
            ax.plot(t, pos, ".", label="Position", color="b")
            ax.plot(
                t[t_beg_mask],
                pos[t_beg_mask],
                "o",
                label="Beg reach t_beg_mask",
                color="g",
            )
            ax.plot(
                t[t_end_mask],
                pos[t_end_mask],
                "o",
                label="End reach t_end_mask",
                color="r",
            )

            ax.plot(
                t_beg,
                np.median(beg_position),
                "*",
                label="Beg reach median",
                color="g",
                markersize=20,
                markeredgewidth=2,
                markeredgecolor="k",
            )
            ax.plot(
                t_end,
                np.median(end_position),
                "*",
                label="End reach median",
                color="r",
                markersize=20,
                markeredgewidth=2,
                markeredgecolor="k",
            )

            ax.vlines(
                t_beg,
                ymin=np.min(pos),
                ymax=np.max(pos),
                color="g",
                linestyle="--",
                label="Beg reach",
            )
            ax.vlines(
                t_end,
                ymin=np.min(pos),
                ymax=np.max(pos),
                color="r",
                linestyle="--",
                label="End reach",
            )

            ax.hlines(
                np.median(beg_position),
                xmin=t_beg,
                xmax=t_end,
                color="g",
                linestyle="--",
            )
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Distance from the kinect (m)")

        # plot the data with 2 subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

        plot_one_sub(ax1)
        ax1.set_title("Whole reach series")
        plot_one_sub(ax2)
        t_zoom = t_beg - 2, t_end + 2
        ax2.set_xlim(t_zoom)
        # ax2.set_title("Zoom on the reach of interest")
        from matplotlib.patches import ConnectionPatch

        xy1 = (float(t_end), float(np.min(pos)))
        xy2 = (float(t_end), float(np.max(pos)))
        connection_end = ConnectionPatch(
            xyA=xy1,
            xyB=xy2,
            coordsA="data",
            coordsB="data",
            axesA=ax1,
            axesB=ax2,
            color="red",
        )
        ax2.add_artist(connection_end)

        xy1 = (t_beg, np.min(pos))
        xy2 = (t_beg, np.max(pos))
        connection_beg = ConnectionPatch(
            xyA=xy1,
            xyB=xy2,
            coordsA="data",
            coordsB="data",
            axesA=ax1,
            axesB=ax2,
            color="green",
        )
        ax2.add_artist(connection_beg)

        plt.show()

    return {
        "beg_position": np.median(beg_position),
        "end_position": np.median(end_position),
        "t_beg": t_beg,
        "t_end": t_end,
        "t_beg_i": t_beg_i,
        "t_end_i": t_end_i,
    }

In [ ]:
## get the reaches in the reaching time zone
def get_reaches(t, wrist, wrist_f):
    """get the reaches on this wrist"""
    find_peaks_threshold = np.median(wrist_f) - 0.1

    # find_peaks_threshold = threshold_median_iqr

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    inter_peaks_time = 60  # 2 seconds
    peaks, _ = find_peaks(
        -wrist_f, height=-find_peaks_threshold, distance=inter_peaks_time
    )

    if len(peaks) == 0:
        print("No peaks found")
        return None, None, None

    # remove the peaks that are outliers in the filtered data
    reaches_end = wrist_f[peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    peaks = np.delete(peaks, i_outliers)

    # get the reaches (t_beg, t_end)
    reaches = []
    for pk in peaks:
        reach = get_one_reach(t, wrist, t_end_i=pk, do_plot=False)
        if reach:
            reaches.append(reach)

    return peaks, reaches, find_peaks_threshold


#########################################################################################
# re-read the data (that may have been modified)
kinect_t = kinect_mocap["time_stamps"]
kinect_data = kinect_mocap["time_series"]

WristLeft_Norm = get_joint_norm(kinect_mocap, "WristLeft")
WristRight_Norm = get_joint_norm(kinect_mocap, "WristRight")

WristLeft_Norm_f = lowpass_filter(kinect_t, WristLeft_Norm)
WristRight_Norm_f = lowpass_filter(kinect_t, WristRight_Norm)

peaks_left, reaches_left, thresh_left = get_reaches(
    kinect_t, WristLeft_Norm, WristLeft_Norm_f
)
peaks_right, reaches_right, thresh_right = get_reaches(
    kinect_t, WristRight_Norm, WristRight_Norm_f
)

if do_visualize:

    def plot_reaches(
        ax, t, wrist, wrist_f, i_peaks, reaches, label="wrist", color="b", thresh=None
    ):
        """ " Plot the reaches of the wrist"""
        ax.plot(t, wrist, ".", label=label, color=color)
        ax.plot(t, wrist_f, label=f"{label} filtered", color=color, alpha=0.2)
        ax.plot(
            t[i_peaks],
            wrist_f[i_peaks],
            "x",
            color="r",
        )
        # plot the threshold
        if thresh is not None:
            ax.axhline(
                y=thresh,
                color=color,
                linestyle="--",
                label=f"Threshold {label}",
            )
        for reach in reaches:
            ax.plot(
                reach["t_beg"],
                reach["beg_position"],
                "o",
                color="orange",
            )
            ax.plot(
                reach["t_end"],
                reach["end_position"],
                "o",
                color="r",
            )
            ax.plot(
                [reach["t_beg"], reach["t_end"]],
                [reach["beg_position"], reach["end_position"]],
                color="k",
                linestyle="--",
            )

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance from the kinect (m)")
        ax.legend()

    # plot the signal and the peaks
    fig, ax = plt.subplots(figsize=(10, 5))
    if peaks_left is not None:
        plot_reaches(
            ax,
            kinect_t,
            WristLeft_Norm,
            WristLeft_Norm_f,
            peaks_left,
            reaches_left,
            label="Left wrist",
            color="b",
            thresh=thresh_left,
        )
    if peaks_right is not None:
        plot_reaches(
            ax,
            kinect_t,
            WristRight_Norm,
            WristRight_Norm_f,
            peaks_right,
            reaches_right,
            label="Right wrist",
            color="k",
            thresh=thresh_right,
        )

    plt.show()

In [ ]:
class KinectJoint:
    """3D kinect joint, made of either 3 ndarray of coordinates x,y,z or 1 ndarray of 3D coordinates xyz"""

    def __init__(self, name, x_or_xyz, y=None, z=None):
        if isinstance(x_or_xyz, np.ndarray) and x_or_xyz.ndim == 2:
            self.xyz = x_or_xyz
        elif isinstance(x_or_xyz, np.ndarray) and x_or_xyz.ndim == 1:
            if y is None or z is None:
                raise ValueError("If x is a 1D array, y and z must be provided")
            self.xyz = np.array([x_or_xyz, y, z]).T
        else:
            raise ValueError("x must be a 1D or 2D array")
        if self.xyz.ndim != 2 or self.xyz.shape[1] != 3:
            raise ValueError("xyz must be a 2D array with 3 columns")

        self.name = name
        self.x = self.xyz[:, 0]
        self.y = self.xyz[:, 1]
        self.z = self.xyz[:, 2]

    def __repr__(self):
        """Get the string representation of the joint"""
        return f"KinectJoint({self.name}, {self.xyz.shape[0]} samples)"

    def __str__(self):
        """Get the string representation of the joint"""
        return f"KinectJoint({self.name}, {self.xyz.shape[0]} samples)"


left_wrist = KinectJoint("LeftWrist", WristLeft_X, WristLeft_Y, WristLeft_Z)
print(left_wrist)

xyz = np.array([WristLeft_X, WristLeft_Y, WristLeft_Z]).T
left_wrist = KinectJoint("LeftWrist", xyz)
print(left_wrist)

In [ ]:
def get_median_data_around_timestamp(data, time_stamps, t_stamp):
    """Get the median data around the timestamp t_stamp"""

    # find the index of the closest timestamp
    i_data = np.argmin(np.abs(time_stamps - t_stamp))

    # we want the median of the positions at i_data +/- 0.5 seconds
    # we assume that we have enough data before and after the index
    half_window = 15
    mask = range(i_data - half_window, i_data + half_window)
    data = data[mask]
    median_data = np.median(data, axis=0)

    return median_data


def get_median_xyz(kinect_mocap, reach_median, channel_name_X):
    """Get the x, y, z coordinates of a kinect channel in reach_median"""
    
    i_channel = get_kinect_channel_index(kinect_mocap, channel_name_X)
    median_xyz = reach_median[i_channel : i_channel + 3]
    return median_xyz

if reaches_left and reaches_right:
    print(f"first_reach_left_start time {reaches_left[0]["t_beg"]}")
    first_reach_left_start = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_left[0]["t_beg"]
    )

    print(f"first_reach_left_end time {reaches_left[0]["t_end"]}")
    first_reach_left_end = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_left[0]["t_end"]
    )

    print(f"first_reach_right_start time {reaches_right[0]["t_beg"]}")
    first_reach_right_start = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_right[0]["t_beg"]
    )
    print(f"first_reach_right_end time {reaches_right[0]["t_end"]}")
    first_reach_right_end = get_median_data_around_timestamp(
        kinect_mocap["time_series"], 
        kinect_mocap["time_stamps"],
        reaches_right[0]["t_end"]
    )


    first_reach_left_start_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_left_start, "WristLeft_X"
    )
    first_reach_left_end_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_left_end, "WristLeft_X"
    )

    first_reach_right_start_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_right_start, "WristRight_X"
    )

    first_reach_right_end_wrist_xyz = get_median_xyz(
        kinect_mocap, first_reach_right_end, "WristRight_X"
    )


    print(
        f"Left wrist position at the beginning of the first reach: {first_reach_left_start_wrist_xyz}"
    )
    print(
        f"Left wrist position at the end of the first reach: {first_reach_left_end_wrist_xyz}"
    )
    print(
        f"Right wrist position at the beginning of the first reach: {first_reach_right_start_wrist_xyz}"
    )
    print(
        f"Right wrist position at the end of the first reach: {first_reach_right_end_wrist_xyz}"
    )

    first_reach_left_distance = np.linalg.norm(
        first_reach_left_end_wrist_xyz - first_reach_left_start_wrist_xyz
    )
    first_reach_right_distance = np.linalg.norm(
        first_reach_right_end_wrist_xyz - first_reach_right_start_wrist_xyz
    )
    print(f"Left wrist distance: {first_reach_left_distance:.2f} m")
    print(f"Right wrist distance: {first_reach_right_distance:.2f} m")

In [ ]:
if reaches_left and reaches_right:
    # get the reaches by block

    # plot the reach times
    reach_left_times = np.array([reach["t_beg"] for reach in reaches_left])
    reach_right_times = np.array([reach["t_beg"] for reach in reaches_right])
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(reach_left_times, np.zeros_like(reach_left_times), "o", label="Left wrist")
    ax.plot(
        reach_right_times, np.ones_like(reach_right_times), "o", label="Right wrist"
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Wrist")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Left wrist", "Right wrist"])
    ax.legend()
    plt.show()

    # make a single reach list from the left and right
    def get_reach_list(reaches_left, reaches_right):
        """Get the reach list from the left and right reaches"""
        reaches = []
        for reach in reaches_left:
            reach["wrist"] = "left"
            reaches.append(reach)
        for reach in reaches_right:
            reach["wrist"] = "right"
            reaches.append(reach)
        return reaches

    reaches = get_reach_list(reaches_left, reaches_right)

    # sort the reaches by time
    reaches = sorted(reaches, key=lambda x: x["t_beg"])
    # print the reaches
    for i, reach in enumerate(reaches):
        print(
            f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}"
        )

    # find the switches between left and right
    def find_switches(reaches):
        """Find the switches between left and right reaches"""
        switches = []
        for i in range(len(reaches) - 1):
            if reaches[i]["wrist"] != reaches[i + 1]["wrist"]:
                switches.append(i)
        return switches

    switches = find_switches(reaches)
    # print the switches
    for i in switches:
        print(
            f"Switch at {i:02d}: {reaches[i]['t_beg']:8.2f} -> {reaches[i]['t_end']:8.2f}"
        )

    # plot the reaches with the switches
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, reach in enumerate(reaches):
        if reach["wrist"] == "left":
            color = "b"
        else:
            color = "k"
        ax.plot(
            [reach["t_beg"], reach["t_end"]],
            [reach["beg_position"], reach["end_position"]],
            color=color,
            linestyle="--",
        )
        ax.plot(
            reach["t_beg"],
            reach["beg_position"],
            "o",
            color=color,
        )
        ax.plot(
            reach["t_end"],
            reach["end_position"],
            "o",
            color=color,
        )
        # plot the switches
        if i in switches:
            ax.plot(
                reach["t_beg"],
                reach["beg_position"],
                "x",
                color="r",
                markersize=10,
                label="Switch",
            )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()
    plt.show()

    # the calibration reaches are the contiguous switches
    i_calibration_reaches = np.where(np.diff(switches) == 1)[0]
    # append the next index (as a switch implies two hands)
    i_calibration_reaches = np.append(i_calibration_reaches, i_calibration_reaches + 1)

    # print the reaches in identifying the calibration reaches
    for i, reach in enumerate(reaches):
        msg = ""
        if i in i_calibration_reaches:
            msg = ", Calibration reach"
            print(
                f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist'] }{msg}"
            )

In [ ]:
i_ch = get_kinect_channel_indexes(kinect_mocap, "Shoulder")

# print the channel names
for i in i_ch:
    print(
        f"{i}: {kinect_mocap['info']['desc'][0]['channels'][0]['channel'][i]['label'][0]}"
    )

In [ ]:
# get the start and stop of the conditions from the markers


def get_start_stop_times_markers():
    if mouse_markers:
        start_times_from_mouse_markers, stop_times_from_mouse_markers = (
            get_start_stop_times_from_mouse_markers(mouse_markers)
        )
        # print("Mouse markers:")
        print_start_stop_times(
            (start_times_from_mouse_markers, stop_times_from_mouse_markers)
        )
        return start_times_from_mouse_markers, stop_times_from_mouse_markers

    if event_to_nic_markers:
        start_times_from_event_ide_tonic, stop_times_from_event_ide_tonic = (
            get_start_stop_times_from_event_ide_TONIC(event_to_nic_markers)
        )
        # print("Event IDE to NIC markers:")
        print_start_stop_times(
            (start_times_from_event_ide_tonic, stop_times_from_event_ide_tonic)
        )
        return start_times_from_event_ide_tonic, stop_times_from_event_ide_tonic

    return None, None


start_times, stop_times = get_start_stop_times_markers()

print_start_stop_times((start_times, stop_times))

# plot the right and left wrist distances with the start and stop times
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    kinect_mocap["time_stamps"],
    WristLeft_Norm,
    ".",
    label="Left wrist ",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    WristRight_Norm,
    ".",
    label="Right wrist ",
    color="k",
)

if start_times is not None and stop_times is not None:
    for i, (start, stop) in enumerate(zip(start_times, stop_times)):
        ax.vlines(
            start,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="g",
            linestyle="--",
        )
        ax.vlines(
            stop,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="r",
            linestyle="--",
        )

ax.legend()
plt.show()

In [ ]:
condition_names = ["sau-p", "sau-np", "mau-p", "mau-np"]

conditions = []
for i, name in enumerate(condition_names):
    if i == len(condition_names) - 1:
        t_end = reaches[-1]["t_end"]
    else:
        t_end = reaches[switches[i + 2]]["t_beg"]

    condition = {
        "name": name,
        "i_switch": i + 1,
        "start": reaches[switches[i + 1]]["t_end"],  # after the end of the switch
        "end": t_end,
    }
    conditions.append(condition)


# print the conditions
for i, condition in enumerate(conditions):
    print(
        f"{i:02d}: {condition['name']:6s}, Switch: {condition['i_switch']:2d}, Start: {condition['start']:8.2f} to {condition['end']:8.2f}, Duration: {condition['end'] - condition['start']:5.2f}s"
    )


# plot the conditions
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    kinect_mocap["time_stamps"],
    WristLeft_Norm,
    ".",
    label="Left wrist ",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    WristLeft_Norm,
    ".",
    label="Right wrist ",
    color="k",
)
for i, condition in enumerate(conditions):
    ax.vlines(
        condition["start"],
        ymin=np.min(WristLeft_Norm),
        ymax=np.max(WristLeft_Norm),
        color="g",
        linestyle="--",
    )
    ax.vlines(
        condition["end"],
        ymin=np.min(WristLeft_Norm),
        ymax=np.max(WristLeft_Norm),
        color="r",
        linestyle="--",
    )

    ax.hlines(
        i,
        xmin=condition["start"],
        xmax=condition["end"],
        color="g",
        linestyle="--",
    )

    ax.text(
        (condition["start"] + condition["end"]) / 2,
        i + 0.1,
        condition["name"],
        ha="center",
        va="bottom",
        # move to front with color = red
        color="red",
        zorder=10,
    )
ax.set_xlabel("Time (s)")
ax.set_ylabel("Condition")
plt.legend()
plt.show()

In [ ]:
paretic = "left"

if start_times is not None and stop_times is not None:
    is_valid = np.array([False] * len(start_times))
    is_paretic = np.array([False] * len(start_times))
    nb_reaches_p = np.array([0] * len(start_times))
    nb_reaches_np = np.array([0] * len(start_times))
    labels = np.array(["unknown"] * len(start_times), dtype="<U15")
    # for each start-stop pair, count the number of reaches
    for i, (start, stop) in enumerate(zip(start_times, stop_times)):
        for reach in reaches:
            if reach["t_beg"] >= start and reach["t_end"] <= stop:
                if reach["wrist"] == paretic:
                    nb_reaches_p[i] += 1
                else:
                    nb_reaches_np[i] += 1

        if nb_reaches_p[i] >= 2:
            is_valid[i] = True
            is_paretic[i] = True
            labels[i] = "Paretic"

        if nb_reaches_np[i] >= 2:
            is_valid[i] = True
            is_paretic[i] = False
            labels[i] = "Non-paretic"

    # get the valid zones
    valid_start_times = start_times[is_valid]
    valid_stop_times = stop_times[is_valid]
    valid_nb_reaches_p = nb_reaches_p[is_valid]
    valid_nb_reaches_np = nb_reaches_np[is_valid]
    valid_is_paretic = is_paretic[is_valid]
    valid_labels = labels[is_valid]

    # print the valid zones
    print("Valid start and stop times:")
    for i, (start, stop, lab) in enumerate(
        zip(valid_start_times, valid_stop_times, valid_labels)
    ):
        print(
            f"{i:02d}: {start:8.2f} -> {stop:8.2f}, Duration: {stop - start:5.2f}s, {lab}"
        )

    # plot the valid zones
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        kinect_mocap["time_stamps"],
        WristLeft_Norm,
        ".",
        label="Left wrist ",
        color="b",
    )
    ax.plot(
        kinect_mocap["time_stamps"],
        WristRight_Norm,
        ".",
        label="Right wrist ",
        color="k",
    )
    for i, (start, stop) in enumerate(zip(valid_start_times, valid_stop_times)):
        ax.vlines(
            start,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="g",
            linestyle="--",
        )
        ax.vlines(
            stop,
            ymin=np.min(WristLeft_Norm),
            ymax=np.max(WristLeft_Norm),
            color="r",
            linestyle="--",
        )
        # add a label to the zone
        y_pos = np.min(WristLeft_Norm) - 0.1
        ax.text(
            (start + stop) / 2,
            y_pos + 0.05,  # np.min(left_wrist_distance),
            f"{valid_nb_reaches_np[i]:02d}",
            ha="center",
            va="bottom",
            color="k",
            fontsize=10,
        )
        ax.text(
            (start + stop) / 2,
            y_pos + 0.04,  # np.min(left_wrist_distance),
            f"{valid_nb_reaches_p[i]:02d}",
            ha="center",
            va="top",
            color="blue",
            fontsize=10,
        )

        ylim = ax.get_ylim()
        ax.set_ylim(y_pos, ylim[1])
        ax.legend()
        plt.show()

In [ ]:
def set_sau_mau_zones(valid_labels):
    """Set the SAU and MAU zones from the valid labels"""

    sau = {
        "name": "SAU",
        "start time": valid_start_times[0],
        "end time": -1,
        "nb Paretic": 0,
        "nb Non-paretic": 0,
    }
    mau = {
        "name": "MAU",
        "start time": -1,
        "end time": valid_stop_times[-1],
        "nb Paretic": 0,
        "nb Non-paretic": 0,
    }
    # when the bloc label changes from Non-paretic to Paretic, this is the end of sau
    # as the sequence : sau-p, sau-np, mau-p, mau-np
    i_end_sau = []  # in case we have multiple changes...
    for i in range(len(valid_labels) - 1):
        if valid_labels[i] != valid_labels[i + 1]:
            if valid_labels[i] == "Non-paretic":
                i_end_sau.append(i)

    # if we have more than 1 i_end_sau, take the last one (experimental problem in the previous trials...)
    i_end_SAU = i_end_sau[-1]

    sau["end time"] = valid_stop_times[i_end_SAU]
    mau["start time"] = valid_start_times[i_end_SAU + 1]

    # add 0.5 seconds to the start and end time of sau and mau
    # Reason: lp filter tends to shift the start time of the reach (see reach figures)
    sau["start time"] -= 0.5
    sau["end time"] += 0.5
    mau["start time"] -= 0.5
    mau["end time"] += 0.5

    # add the number of labels in the sau and mau
    for i in range(len(valid_labels)):
        if i <= i_end_SAU:
            if valid_labels[i] == "Non-paretic":
                sau["nb Non-paretic"] += 1
            else:
                sau["nb Paretic"] += 1
        else:
            if valid_labels[i] == "Non-paretic":
                mau["nb Non-paretic"] += 1
            else:
                mau["nb Paretic"] += 1

    # add the duration of the sau and mau
    sau["duration"] = sau["end time"] - sau["start time"]
    mau["duration"] = mau["end time"] - mau["start time"]

    return sau, mau


def print_zone_dict(d):
    """Print a sau-mau-cal dictionary"""
    print(
        f"{d['name']}: {d['start time']:5.2f} -> {d['end time']:.2f}: {d['duration']:7.2f}s, Blocks: {d['nb Paretic']} Paretic, {d['nb Non-paretic']} Non-paretic"
    )


####################################################################################
sau, mau = set_sau_mau_zones(valid_labels)


# add labels to the reaches
for reach in reaches:
    # add the sau and mau labels to the reaches
    if reach["t_beg"] >= sau["start time"] and reach["t_end"] <= sau["end time"]:
        reach["condition"] = "sau"
    elif reach["t_beg"] >= mau["start time"] and reach["t_end"] <= mau["end time"]:
        reach["condition"] = "mau"
    else:
        reach["condition"] = "unknown"
    # add the paretic label to the reaches
    if reach["wrist"] == paretic:
        reach["arm"] = "Paretic"
    else:
        reach["arm"] = "Non-paretic"


reaches_panu = [reach for reach in reaches if reach["condition"] != "unknown"]
reaches_unknown = [reach for reach in reaches if reach["condition"] == "unknown"]


# NOTE: unknown should be the calibration reaches
cal = {
    "name": "CAL",
    "start time": reaches_unknown[0]["t_beg"],
    "end time": reaches_unknown[-1]["t_end"],
    "duration": reaches_unknown[-1]["t_end"] - reaches_unknown[0]["t_beg"],
    "nb Non-paretic": len(
        [reach for reach in reaches_unknown if reach["wrist"] == paretic]
    ),
    "nb Paretic": len(
        [reach for reach in reaches_unknown if reach["wrist"] != paretic]
    ),
}


print_zone_dict(cal)
print_zone_dict(sau)
print_zone_dict(mau)

In [ ]:
def get_calib_xyz(kinect_mocap, reach):
    """Get the calibration xyz of the wrist at the beginning and end of the reach"""

    joint_name = f"Wrist{'Left' if reach['wrist'] == 'left' else 'Right'}"

    median_arround_reach_beg = get_median_data_around_timestamp(
        kinect_mocap["time_series"], kinect_mocap["time_stamps"], reach["t_beg"]
    )
    calib_xyz_at_beg = get_median_xyz(
        kinect_mocap, median_arround_reach_beg, f"{joint_name}_X"
    )

    median_arround_reach_end = get_median_data_around_timestamp(
        kinect_mocap["time_series"], kinect_mocap["time_stamps"], reach["t_end"]
    )
    calib_xyz_at_end = get_median_xyz(
        kinect_mocap, median_arround_reach_end, f"{joint_name}_X"
    )
    # get the median of the two positions

    return {
        "beg": calib_xyz_at_beg,
        "end": calib_xyz_at_end,
        "distance": np.linalg.norm(calib_xyz_at_end - calib_xyz_at_beg),
        "t_beg": reach["t_beg"],
        "t_end": reach["t_end"],
        "joint": joint_name,
    }


# get the reaches in the cal zone
reaches_cal = [reach for reach in reaches if reach["condition"] == "unknown"]

print(f"Number of reaches in the calibration zone: {len(reaches_cal)}")
# print the reaches in the calibration zone
for i, reach in enumerate(reaches_cal):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}, Condition: {reach['condition']}, Arm: {reach['arm']}"
    )

# left wrist calibration
calib_xyz_0 = get_calib_xyz(kinect_mocap, reaches_cal[0])
print(f"{calib_xyz_0['joint']}")
print(f"@beg: {calib_xyz_0['beg']}")
print(f"@end: {calib_xyz_0['end']}")
print(f"dist: {calib_xyz_0['distance']:.4f} m")

# right wrist calibration
calib_xyz_1 = get_calib_xyz(kinect_mocap, reaches_cal[1])
print(f"{calib_xyz_1['joint']}")
print(f"@beg: {calib_xyz_1['beg']}")
print(f"@end: {calib_xyz_1['end']}")
print(f"dist: {calib_xyz_1['distance']:.4f} m")

# plot the calibration reaches
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    kinect_mocap["time_stamps"],
    WristLeft_Norm,
    ".",
    label="Left wrist ",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    WristRight_Norm,
    ".",
    label="Right wrist ",
    color="k",
)
for i, reach in enumerate(reaches_cal):
    if reach["wrist"] == "left":
        color = "b"
    else:
        color = "k"
    ax.plot(
        [reach["t_beg"], reach["t_end"]],
        [reach["beg_position"], reach["end_position"]],
        color=color,
        linestyle="--",
    )
    ax.plot(
        reach["t_beg"],
        reach["beg_position"],
        "o",
        color="g",
    )
    ax.plot(
        reach["t_end"],
        reach["end_position"],
        "o",
        color="r",
    )

ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance from the kinect (m)")
ax.legend()
plt.show()

## Get the target position

In [ ]:
# first reach positions
target_xyz = np.mean(
    [first_reach_left_end_wrist_xyz, first_reach_right_end_wrist_xyz], axis=0
)
print(f"Left wrist first position: {first_reach_left_end_wrist_xyz}")
print(f"Right wrist first position: {first_reach_right_end_wrist_xyz}")
# target_xyz position (m): -0.000 0.008 1.931


# Calibration positions
if calib_xyz_0["joint"] == "WristLeft":
    calib_xyz_left = calib_xyz_0["end"]
    calib_xyz_right = calib_xyz_1["end"]
else:
    calib_xyz_left = calib_xyz_1["end"]
    calib_xyz_right = calib_xyz_0["end"]

print(f"Left wrist calibration position: {calib_xyz_left}")
print(f"Right wrist calibration position: {calib_xyz_right}")
target_xyz = np.mean([calib_xyz_0["end"], calib_xyz_1["end"]], axis=0)
# target_xyz position (m): -0.000 0.008 1.931


# Format the arrays for better readability
txt = ""
for x in target_xyz:
    txt += f"{x:.3f} "
print(f"target_xyz position (m): {txt}")

## Compute the distance to the target, for both wrists and both shoulders

In [ ]:
def get_distance_to_target(xyz, target_xyz):
    """Get the distance to the target"""

    # check that the target is a 1D array of 3 elements
    if target_xyz.ndim != 1 or target_xyz.shape[0] != 3:
        raise ValueError("target_xyz must be a 1D array of 3 elements")

    # check that xyz is either a 1D or 2D array of 3 elements by row
    if xyz.ndim == 1:
        # reshape it to a 2D array (with one row)
        xyz = np.array([xyz])
    elif xyz.ndim == 2:
        # check that it has 3 columns
        if xyz.shape[1] != 3:
            raise ValueError("xyz must be a 2D array of 3 elements")
    else:
        raise ValueError("xyz must be a 1D or 2D array")

    # get the distance to the target
    distance_to_target = np.linalg.norm(xyz - target_xyz, axis=1)

    return distance_to_target


def get_kinect_channel_xyz(kinect_mocap, channel_name_X):
    """Get the x, y, z coordinates of a kinect channel in the kinect_mocap stream"""
    i_channel = get_kinect_channel_index(kinect_mocap, channel_name_X)
    channel_data = kinect_mocap["time_series"][:, i_channel : i_channel + 3]
    return channel_data


left_wrist_xyz = get_kinect_channel_xyz(kinect_mocap, "WristLeft_X")
right_wrist_xyz = get_kinect_channel_xyz(kinect_mocap, "WristRight_X")
left_shoulder_xyz = get_kinect_channel_xyz(kinect_mocap, "ShoulderLeft_X")
right_shoulder_xyz = get_kinect_channel_xyz(kinect_mocap, "ShoulderRight_X")

left_wrist_distance = get_distance_to_target(left_wrist_xyz, target_xyz)
right_wrist_distance = get_distance_to_target(right_wrist_xyz, target_xyz)
left_shoulder_distance = get_distance_to_target(left_shoulder_xyz, target_xyz)
right_shoulder_distance = get_distance_to_target(right_shoulder_xyz, target_xyz)


print(left_wrist_distance.shape)

# plot the left_wrist_distance over time
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    kinect_mocap["time_stamps"],
    left_wrist_distance,
    ".",
    label="Left wrist distance",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    right_wrist_distance,
    ".",
    label="Right wrist distance",
    color="k",
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance to target (m)")
ax.legend()
plt.show()

In [ ]:
# add the PANU ingredients to the reaches
for reach in reaches:
    # if it is a left reach, add the left wrist distance
    if reach["wrist"] == "left":
        reach["wrist_start_distance"] = get_median_data_around_timestamp(
            left_wrist_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )
        reach["wrist_end_distance"] = get_median_data_around_timestamp(
            left_wrist_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )
        # add the shoulder distance
        reach["shoulder_start_distance"] = get_median_data_around_timestamp(
            left_shoulder_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )
        reach["shoulder_end_distance"] = get_median_data_around_timestamp(
            left_shoulder_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )
    # if it is a right reach, add the right wrist distance
    if reach["wrist"] == "right":
        reach["wrist_start_distance"] = get_median_data_around_timestamp(
            right_wrist_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )
        reach["wrist_end_distance"] = get_median_data_around_timestamp(
            right_wrist_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )
        # add the shoulder distance
        reach["shoulder_start_distance"] = get_median_data_around_timestamp(
            right_shoulder_distance, kinect_mocap["time_stamps"], reach["t_beg"]
        )
        reach["shoulder_end_distance"] = get_median_data_around_timestamp(
            right_shoulder_distance, kinect_mocap["time_stamps"], reach["t_end"]
        )

    # for all the reaches, add the traveled distances
    reach["shoulder_travel"] = (
        reach["shoulder_end_distance"] - reach["shoulder_start_distance"]
    )
    reach["wrist_travel"] = reach["wrist_end_distance"] - reach["wrist_start_distance"]
    reach["S/W ratio"] = reach["shoulder_travel"] / reach["wrist_travel"]


# print the content of the reaches
for i, reach in enumerate(reaches):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']:6s}, Travel: wrist {reach['wrist_travel']:.2f} m, shoulder: {reach['shoulder_travel']:.2f} m, ratio {reach['S/W ratio']:.2f}, "
    )

In [ ]:
# plot the valid zones with the SAU and MAU
fig, ax = plt.subplots(figsize=(10, 5))

# plot the left and right wrist distances
ax.plot(
    kinect_mocap["time_stamps"],
    left_wrist_distance,
    ".",
    label="Left wrist distance",
    color="b",
)
ax.plot(
    kinect_mocap["time_stamps"],
    right_wrist_distance,
    ".",
    label="Right wrist distance",
    color="k",
)


for i, (start, stop) in enumerate(zip(valid_start_times, valid_stop_times)):
    ax.vlines(
        start,
        ymin=np.min(left_wrist_distance),
        ymax=np.max(left_wrist_distance),
        color="g",
        linestyle="--",
    )
    ax.vlines(
        stop,
        ymin=np.min(left_wrist_distance),
        ymax=np.max(left_wrist_distance),
        color="r",
        linestyle="--",
    )
    # add the number of reaches in the zone
    ax.text(
        (start + stop) / 2,
        0.02,
        f"{valid_nb_reaches_np[i]:02d}",
        ha="center",
        va="bottom",
        color="k",
        fontsize=10,
    )
    ax.text(
        (start + stop) / 2,
        0.0,
        f"{valid_nb_reaches_p[i]:02d}",
        ha="center",
        va="bottom",
        color="blue",
        fontsize=10,
    )
# add the SAU and MAU as a colored zone
top = np.max([reach["wrist_start_distance"] for reach in reaches]) + 0.01
bot = np.min([reach["wrist_end_distance"] for reach in reaches]) - 0.01


def plot_zone(ax, zone, color):
    """Plot a zone on the graph"""
    ax.fill_betweenx(
        [bot, top],
        zone["start time"],
        zone["end time"],
        color=color,
        alpha=0.2,
        label=zone["name"],
    )


plot_zone(ax, cal, "red")
plot_zone(ax, sau, "orange")
plot_zone(ax, mau, "black")

for i, reach in enumerate(reaches):

    if reach["wrist"] == "left":
        color = "b"
        wrist_data = left_wrist_distance
    else:
        color = "k"
        wrist_data = right_wrist_distance

    # a dotted line from the start to the end of the reach
    ax.plot(
        [reach["t_beg"], reach["t_end"]],
        [reach["wrist_start_distance"], reach["wrist_end_distance"]],
        color=color,
        linestyle="--",
    )
    # a large star at the end
    ax.plot(
        reach["t_end"],
        reach["wrist_end_distance"],
        # make a large star with red border
        markerfacecolor=color,
        markeredgecolor="red",
        markersize=20,
        marker="*",
        # move to the front
        zorder=10,
    )
    # a large star at the start
    ax.plot(
        reach["t_beg"],
        reach["wrist_start_distance"],
        markerfacecolor=color,
        markeredgecolor="green",
        markersize=20,
        marker="*",
        zorder=10,
    )
    # plot the data used to compute the median
    if reach["wrist"] == "left":
        wrist_data = left_wrist_distance
    else:
        wrist_data = right_wrist_distance
    beg_mask = range(reach["t_beg_i"] - 10, reach["t_beg_i"] + 10)
    end_mask = range(reach["t_end_i"] - 10, reach["t_end_i"] + 10)
    ax.plot(
        kinect_mocap["time_stamps"][beg_mask],
        wrist_data[beg_mask],
        "o",
        color="green",
    )
    ax.plot(
        kinect_mocap["time_stamps"][end_mask],
        wrist_data[end_mask],
        "o",
        color="red",
    )
    # print(f"Reach {i:02d}: {reach['t_beg']:.2f} -> {reach['t_end']:.2f} ( {kinect_mocap["time_stamps"][reach["t_beg_i"]]:.2f} -> {kinect_mocap["time_stamps"][reach["t_end_i"]]:.2f} )")

ylim = ax.get_ylim()
ax.set_ylim(0, ylim[1])
ax.legend()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Distance to target (m)")
plt.show()

In [ ]:
# get the reaches in the SAU and MAU
def get_reaches_in_zone(reaches, zone):
    """Get the reaches in the zone"""
    reaches_in_zone = []
    for reach in reaches:
        if reach["t_beg"] >= zone["start time"] and reach["t_end"] <= zone["end time"]:
            reaches_in_zone.append(reach)
    return reaches_in_zone


sau_reaches = get_reaches_in_zone(reaches_panu, sau)
mau_reaches = get_reaches_in_zone(reaches_panu, mau)
# print the reaches in the SAU and MAU
print("SAU reaches:")
for i, reach in enumerate(sau_reaches):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, ratio: {reach['S/W ratio']:.2f} {reach['wrist']}"
    )
print("MAU reaches:")
for i, reach in enumerate(mau_reaches):
    print(
        f"{i:02d}: {reach['t_beg']:8.2f} -> {reach['t_end']:8.2f}, Duration: {reach['t_end'] - reach['t_beg']:5.2f}s, Wrist: {reach['wrist']}"
    )

In [ ]:
# save the reaches in a CSV file
import pandas as pd
import os


def save_reaches_in_csv(reaches, file_name):
    """Save the reaches in a CSV file"""
    # create the directory if it does not exist
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    # create the dataframe
    df = pd.DataFrame(reaches)
    # save the dataframe in a CSV file
    df.to_csv(file_name, index=False)


def save_reaches_in_csv_file_name(reaches, xdf_fullFname):
    """Save the reaches in a CSV file with the file name"""
    # get the file name
    file_name = xdf_fullFname.replace(".xdf", "_reaches.csv")
    save_reaches_in_csv(reaches, file_name)


def load_reaches_from_csv_file_name(xdf_fullFname):
    """Load the reaches from a CSV file with the file name"""
    # get the file name
    file_name = xdf_fullFname.replace(".xdf", "_reaches.csv")
    # load the dataframe from the CSV file
    df = pd.read_csv(file_name)
    # reaches = df.to_dict(orient="records")
    return df


# save the reaches in a CSV file
save_reaches_in_csv_file_name(reaches_left, xdf_fullFname)
save_reaches_in_csv_file_name(reaches_right, xdf_fullFname)

In [ ]:
# read the CSV file
df = load_reaches_from_csv_file_name(xdf_fullFname)

# get the median of the ration by condition
sau_reaches = df[df["condition"] == "sau"]
mau_reaches = df[df["condition"] == "mau"]
sau_median = sau_reaches["S/W ratio"].median()
mau_median = mau_reaches["S/W ratio"].median()
print(f"SAU median: {sau_median:.2f}")
print(f"MAU median: {mau_median:.2f}")

# PANU is SAU - MAU
panu = sau_median - mau_median
print(f"PANU: {panu:.2f}")